<a href="https://colab.research.google.com/github/Rahu378/hardware-aware-gateway/blob/main/notebooks/colab_bootstrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hardware-Aware Gateway. Colab bootstrap

**Set the runtime first:** Runtime > Change runtime type > T4 GPU.
Cell 1 fails deliberately if you forget.


In [ ]:
import torch
assert torch.cuda.is_available(), (
    'No CUDA device. Runtime > Change runtime type > T4 GPU, then rerun.'
)
print(torch.cuda.get_device_name(0))
print('torch     ', torch.__version__)
import triton; print('triton    ', triton.__version__)


## Run everything

One cell: clone, install, correctness, benchmarks, report, archive.

It runs under `set -e`, so the first failure stops it. That matters more
than it sounds. Notebook cells run independently, so a failing benchmark
cell does not stop the download cell below it, and an earlier run of this
notebook raised `ModuleNotFoundError` four times while still handing over an
archive that looked like a completed run.

It also uses absolute paths throughout, because re-running a `%cd` cell
descends into a nested clone and a later relative `zip` then archives a
different checkout than the one just built.


In [ ]:
!curl -sL https://raw.githubusercontent.com/Rahu378/hardware-aware-gateway/main/scripts/colab_run.sh | bash


## Download the archive

Only run this if the cell above reached `Archive: /content/artifacts.zip`.


In [ ]:
from google.colab import files
files.download('/content/artifacts.zip')


---

## Running the steps individually

Useful when a step fails and you want to iterate on it. Run this first:


In [ ]:
%cd /content
!rm -rf hag-run
!git clone -q https://github.com/Rahu378/hardware-aware-gateway.git hag-run
%cd /content/hag-run
!pip install -q -e '.[e2e,dev]'
!python -c "import hag; print('import OK:', hag.__file__)"


Correctness. Do not continue past a failure here; a wrong kernel still
produces fast numbers.


In [ ]:
!python -m pytest -q


...........................sssssssssssssssssssss..                       [100%]


Profile before changing anything. If elementwise kernels (`mul`, `add`,
`silu`, the norms) outrank the GEMMs, the workload is memory-bound and
fusion is the right lever.


In [ ]:
!python -m hag.profile_torch --model Qwen/Qwen2.5-1.5B --prompt-tokens 512 --new-tokens 32


Op-level sweep. Writes `results/ops_nvidia-t4_fp16.json`.


In [ ]:
!python -m hag.bench_ops --backend cuda --dtype fp16


End-to-end. The number that decides whether the kernel work mattered.


In [ ]:
!python -m hag.bench_e2e --model Qwen/Qwen2.5-1.5B --prompt-tokens 512 --new-tokens 128


Regenerate the tables. T4 rows should now sit beside the Apple M3 ones.


In [ ]:
!python -m hag.report
!sed -n '/BENCH:BEGIN/,/BENCH:END/p' README.md
!ls -la /content/hag-run/results


In [ ]:
!curl -sL https://raw.githubusercontent.com/Rahu378/hardware-aware-gateway/main/scripts/colab_run.sh | bash

=== 1/6  clone into /content/hag-run ===
=== 2/6  install ===
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 113.9 MB/s eta 0:00:00
  Building editable for hardware-aware-gateway (pyproject.toml) ... done
import OK: /content/hag-run/src/hag/__init__.py
=== 3/6  correctness ===
.....................sssssssssssssssssssss..                             [100%]
=== 4/6  op-level sweep ===
device : Tesla T4  [cuda]
copy   : 241.3 GB/s measured
floor  : 6 us per dispatch

op                         shape   base ms  fused ms  speedup    GB/s  %peak
rmsnorm_residual          1x2048    0.1029    0.0496    2.08x       0     0%
rmsnorm_residual          8x2048    0.1046    0.0518    2.02x       2     1%
rmsnorm_residual        512x2048    0.2432    0.0389    6.25x     216    89%
rmsno

In [ ]:
from google.colab import files
files.download('/content/artifacts.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

### Next: Nsight Compute counters

`ncu` fails on Colab with `ERR_NVGPUCTRPERM`; the runtime does not grant
performance-counter access. That is the one step needing a GCP or Azure VM
on signup credits. See `scripts/profile_ncu.sh` for the module-parameter fix.

`nsys` is optional and not installed here. `hag.profile_torch` answers the
same question with no apt package.
